# Tableau Calculated Fields

**Tableau** is one of the most widely used BI tools. **Tableau Public** is free: it runs on Windows and macOS and publishes workbooks to the public Tableau Public website. Most of Tableau's analytical power lives in **calculated fields**: row-level formulas, aggregations, **level-of-detail (LOD) expressions** and **table calculations**.

Tableau cannot run inside a notebook. This lesson therefore:

1. exports a clean extract of the taxi data for you to open in Tableau Public,
2. explains each kind of calculated field with its Tableau syntax,
3. computes the same result in pandas, as an **answer key** to check your Tableau numbers.

**After this lesson you can:**

1. Write row-level, aggregate and logical calculations in Tableau syntax.
2. Use LOD expressions (`FIXED`, `INCLUDE`, `EXCLUDE`) and explain how they differ.
3. Use table calculations (running totals, percent of total, rank) and parameters.
4. Explain Tableau's order of operations and why it changes results.

**Data:** NYC yellow taxi trips, January 2024 (cleaned), aggregated to one row per pickup zone, day and hour.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "tools"))
import datasets                                   # course helper: downloads pinned public datasets

plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 100})

COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "trip_distance",
        "fare_amount", "tip_amount", "total_amount", "payment_type"]

def load_month(month: str) -> pd.DataFrame:
    """One month of NYC yellow taxi trips with the cleaning rules of M08 Lesson 08."""
    df = pd.read_parquet(datasets.taxi(month), columns=COLS)
    start = pd.Timestamp(month + "-01")
    dur = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    ok = (df["tpep_pickup_datetime"].between(start, start + pd.offsets.MonthBegin(1), inclusive="left")
          & dur.between(1, 180) & df["trip_distance"].between(0.1, 100) & df["fare_amount"].between(3, 500))
    return df[ok].assign(duration_min=dur[ok])

zones = pd.read_csv(datasets.fetch("taxi_zone_lookup.csv"))

EXPORTS = Path.cwd().parent / "data" / "cache" / "exports"          # git-ignored folder for files you open in BI tools
EXPORTS.mkdir(parents=True, exist_ok=True)

trips = load_month("2024-01")
is_card = trips["payment_type"] == 1
extract = (trips.assign(date=trips["tpep_pickup_datetime"].dt.date, hour=trips["tpep_pickup_datetime"].dt.hour,
                        card=is_card, card_fare=trips["fare_amount"].where(is_card, 0.0))
           .groupby(["PULocationID", "date", "hour"])
           .agg(trips=("fare_amount", "size"), fare=("fare_amount", "sum"), tips=("tip_amount", "sum"),
                card_trips=("card", "sum"), card_fare=("card_fare", "sum"), minutes=("duration_min", "sum"))
           .reset_index()
           .merge(zones.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left"))
extract.to_csv(EXPORTS / "taxi_zone_day_hour_2024-01.csv", index=False)
print(f"{len(trips):,} trips -> {len(extract):,} rows exported to data/cache/exports/taxi_zone_day_hour_2024-01.csv")
extract.head(3)

2,855,769 trips -> 73,171 rows exported to data/cache/exports/taxi_zone_day_hour_2024-01.csv


,PULocationID,date,hour,trips,fare,tips,card_trips,card_fare,minutes,Borough,Zone,service_zone
0,1,2024-01-02,18,1,9.0,0.0,1,9.0,1.216667,EWR,Newark Airport,EWR
1,1,2024-01-02,19,1,86.0,0.0,1,86.0,38.633333,EWR,Newark Airport,EWR
2,1,2024-01-03,12,1,91.6,10.0,1,91.6,36.950000,EWR,Newark Airport,EWR


**Try it in Tableau Public:** *Connect → Text file →* `taxi_zone_day_hour_2024-01.csv`. Tableau sorts the fields into **dimensions** (Borough, Zone, date, hour) and **measures** (trips, fare, tips, ...). The extract is already aggregated (one row per zone, day and hour), so **always sum** the measures, never average them across rows.

## 1. Row-Level and Aggregate Calculations

| Kind | Tableau | Evaluated |
| :--- | :--- | :--- |
| row level | `[fare] / [trips]` | for each row of the data source |
| aggregate | `SUM([fare]) / SUM([trips])` | after grouping by the dimensions in the view |

The two give **different** answers: averaging per-row ratios weights every row equally; the ratio of sums weights every trip equally. For "average fare per trip", the ratio of sums is correct.

In [2]:
row_level = (extract["fare"] / extract["trips"]).groupby(extract["Borough"]).mean()          # AVG([fare] / [trips])
aggregate = extract.groupby("Borough")["fare"].sum() / extract.groupby("Borough")["trips"].sum()   # SUM([fare]) / SUM([trips])
pd.DataFrame({"AVG(row ratio)": row_level, "SUM/SUM (correct)": aggregate}).round(2).sort_values("SUM/SUM (correct)")

,AVG(row ratio),SUM/SUM (correct)
Borough,,
Manhattan,17.26,14.91
Unknown,19.92,18.60
Brooklyn,29.13,28.87
Bronx,32.33,32.37
Queens,38.71,52.85
Staten Island,56.85,56.85
EWR,74.61,74.61


## 2. Logical Calculations

```text
// Tableau: Time of Day
IF [hour] < 6 THEN "Night"
ELSEIF [hour] < 12 THEN "Morning"
ELSEIF [hour] < 18 THEN "Afternoon"
ELSE "Evening"
END

// Tableau: Tip Rate (aggregate, safe against division by zero)
IF SUM([card_fare]) > 0 THEN SUM([tips]) / SUM([card_fare]) END

// Tableau: Weekend
DATEPART('weekday', [date]) IN (1, 7)          // 1 = Sunday, 7 = Saturday in the default US calendar
```

In [3]:
extract["time_of_day"] = pd.cut(extract["hour"], [-1, 5, 11, 17, 23], labels=["Night", "Morning", "Afternoon", "Evening"])
extract["weekend"] = pd.to_datetime(extract["date"]).dt.day_of_week >= 5
tip_rate = extract.groupby(["time_of_day", "weekend"], observed=True).apply(
    lambda g: g["tips"].sum() / g["card_fare"].sum(), include_groups=False).unstack().round(4)
print("Tip Rate by Time of Day and Weekend (answer key):")
tip_rate

Tip Rate by Time of Day and Weekend (answer key):


weekend,False,True
time_of_day,,
Night,0.2185,0.2314
Morning,0.2202,0.2306
Afternoon,0.2325,0.2286
Evening,0.2424,0.2295


## 3. Level-of-Detail (LOD) Expressions

A normal aggregate is computed at the **level of the view** (the dimensions on rows, columns, colour, ...). An LOD expression computes an aggregate at a level **you** choose:

| LOD | Meaning | Example |
| :--- | :--- | :--- |
| `{FIXED [Zone] : SUM([trips])}` | at exactly this level, ignoring the view | total trips of each zone, on every row of that zone |
| `{INCLUDE [date] : SUM([trips])}` | the view's level **plus** a dimension | daily trips, then averaged in the view |
| `{EXCLUDE [hour] : SUM([trips])}` | the view's level **minus** a dimension | the zone's daily total in an hourly view |

**Classic uses:** share of a parent total, average of a daily total, cohort first dates (`{FIXED [Customer] : MIN([Order Date])}`).

In [4]:
# {FIXED [Zone] : SUM([trips])} -> one value per zone, attached to every row of the zone
extract["zone_total"] = extract.groupby("Zone")["trips"].transform("sum")
# view: Borough. Measure: AVG({INCLUDE [date] : SUM([trips])}) -> average daily trips per borough
avg_daily = extract.groupby(["Borough", "date"])["trips"].sum().groupby("Borough").mean()
# view: Zone x hour. Measure: SUM([trips]) / SUM({EXCLUDE [hour] : SUM([trips])}) -> share of the zone's trips in each hour
zone_hour = extract.groupby(["Zone", "hour"])["trips"].sum()
share_of_zone = zone_hour / zone_hour.groupby("Zone").transform("sum")
print("average daily trips per borough (INCLUDE):\n", avg_daily.round(0).sort_values(ascending=False).head(4), "\n")
print("JFK Airport, share of its trips by hour (EXCLUDE), 14-17h:\n", share_of_zone.loc["JFK Airport"].loc[14:17].round(4))

average daily trips per borough (INCLUDE):
 Borough
Manhattan    82682.0
Queens        8226.0
Brooklyn       707.0
Unknown        305.0
Name: trips, dtype: float64 

JFK Airport, share of its trips by hour (EXCLUDE), 14-17h:
 hour
14    0.0569
15    0.0636
16    0.0770
17    0.0694
Name: trips, dtype: float64


## 4. Table Calculations and Parameters

**Table calculations** are computed **last**, on the aggregated table that the view shows: `RUNNING_SUM`, `WINDOW_AVG`, `RANK`, `PERCENT_OF_TOTAL`, `LOOKUP` (previous value). "Compute using" decides the direction (across days, down zones, ...).

```text
RUNNING_SUM(SUM([trips]))                       // compute using: date
WINDOW_AVG(SUM([trips]), -6, 0)                 // 7-day moving average
SUM([trips]) / TOTAL(SUM([trips]))              // percent of total
RANK(SUM([trips]))                              // rank of each zone
[Rank] <= [Top N]                               // filter with an integer parameter "Top N"
```

In [5]:
daily = extract.groupby("date")["trips"].sum()
table_calc = pd.DataFrame({"trips": daily, "RUNNING_SUM": daily.cumsum(),
                           "WINDOW_AVG(-6,0)": daily.rolling(7, min_periods=1).mean().round(0),
                           "PERCENT_OF_TOTAL": (daily / daily.sum()).round(4)})
print(table_calc.head(8), "\n")
top_n = 5                                                                   # the parameter
zone_rank = extract.groupby("Zone")["trips"].sum().rank(ascending=False, method="min").sort_values()
print(f"Top {top_n} zones (RANK <= [Top N]):", zone_rank[zone_rank <= top_n].index.tolist())

            trips  RUNNING_SUM  WINDOW_AVG(-6,0)  PERCENT_OF_TOTAL
date                                                              
2024-01-01  74894        74894           74894.0            0.0262
2024-01-02  72603       147497           73748.0            0.0254
2024-01-03  79566       227063           75688.0            0.0279
2024-01-04  99754       326817           81704.0            0.0349
2024-01-05  99717       426534           85307.0            0.0349
2024-01-06  93251       519785           86631.0            0.0327
2024-01-07  64817       584602           83515.0            0.0227
2024-01-08  77410       662012           83874.0            0.0271 

Top 5 zones (RANK <= [Top N]): ['Upper East Side South', 'Midtown Center', 'JFK Airport', 'Upper East Side North', 'Midtown East']


## 5. Order of Operations

Tableau applies filters and calculations in a fixed order. The most important part:

1. extract and data-source filters
2. context filters
3. **`FIXED` LOD expressions**
4. **dimension filters** (the normal filters on the Filters shelf)
5. `INCLUDE` / `EXCLUDE` LOD expressions
6. measure filters
7. **table calculations** and table-calculation filters

**Consequence:** a `FIXED` expression ignores normal dimension filters, unless the filter is added to **context**. The cell reproduces the classic surprise: the share of a borough's trips computed with `FIXED` vs with `EXCLUDE` when a filter keeps only the evening hours.

In [6]:
evening = extract[extract["hour"] >= 18]                                   # dimension filter: hour >= 18
fixed_total = extract.groupby("Borough")["trips"].sum()                     # FIXED [Borough]: computed BEFORE the filter
exclude_total = evening.groupby("Borough")["trips"].sum()                   # EXCLUDE: computed AFTER the filter
zone_trips = evening.groupby(["Borough", "Zone"])["trips"].sum()
compare = pd.DataFrame({
    "share via FIXED (all-day total)": zone_trips / zone_trips.index.get_level_values("Borough").map(fixed_total).to_numpy(),
    "share via EXCLUDE (evening total)": zone_trips / zone_trips.index.get_level_values("Borough").map(exclude_total).to_numpy(),
}).loc["Manhattan"].nlargest(3, "share via EXCLUDE (evening total)").round(4)
compare

,share via FIXED (all-day total),share via EXCLUDE (evening total)
Zone,,
Midtown Center,0.0206,0.0634
Times Sq/Theatre District,0.0170,0.0523
Lincoln Square East,0.0160,0.0492


With `FIXED`, the evening zone trips are divided by the **all-day** borough total, so the shares do not sum to 100 % in the filtered view. With `EXCLUDE` (or with the filter added to context), they are divided by the evening total.

*Insight: Before you write a Tableau calculation, decide at which level it must be computed: row, view, a fixed level (LOD), or on the finished table (table calculation), and remember the order of operations when filters are involved. Check the result against an independent computation, as in this lesson.*

## Real-World Use Case or Analogy

LOD expressions are like asking a **school statistics office** different questions: "the average grade in this class" (the view level), "the average of each student's best grade" (`INCLUDE` student), or "this class's share of the school total" (`FIXED` school). The same data gives different answers depending on the level of the question.

Tableau developers spend much of their time on exactly these calculations; LOD expressions and table calculations are the most common topics in Tableau certification exams and job interviews.